# N17 — Radio Data Labeling

To train RoBERTa and compare results with VADER, I'll label the dataset in this notebook. Post-race radio messages are filtered out to avoid biasing the models — if the dataset becomes too small after filtering, more races are added.

After running this notebook, re-run **N19** (`N19_sentiment_vader.ipynb`) to get results on the cleaned data.

## Step 0 — Setup

In [1]:
# ── Step 0 · Setup ────────────────────────────────────────────────────────────
import sys
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Paths ──────────────────────────────────────────────────────────────────────
PROC_DIR = repo_root / "data" / "processed" / "radio_nlp"
OUTPUTS  = repo_root / "notebooks" / "nlp" / "outputs"

for d in [PROC_DIR, OUTPUTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"repo_root : {repo_root}")
print(f"PROC_DIR  : {PROC_DIR}")
print(f"OUTPUTS   : {OUTPUTS}")

repo_root : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
PROC_DIR  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\radio_nlp
OUTPUTS   : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\nlp\outputs


### First, uploading the csv 

First I need to upload the `radios_raw.csv`.

In [2]:
def load_raw_data(proc_dir):
    df = pd.read_csv(proc_dir / "radios_raw.csv")
    df = df.reset_index(drop=True)
    print(f"Loaded {len(df)} radio messages")
    return df


df = load_raw_data(PROC_DIR)
df.head()

Loaded 659 radio messages


,driver,filename,file_path,text,duration
0,driver_1,driver_1_belgium_radio_39.mp3,c:\Users\victo\Desktop\Documents\Cuarto Año\TF...,"So don't forget Max, use your head please. Are...",15.168
1,driver_1,driver_1_belgium_radio_40.mp3,c:\Users\victo\Desktop\Documents\Cuarto Año\TF...,"Okay Max, we're expecting rain in about 9 or 1...",15.576
2,driver_1,driver_1_belgium_radio_60.mp3,c:\Users\victo\Desktop\Documents\Cuarto Año\TF...,$%&&S& $%&!,5.424
3,driver_1,driver_1_belgium_radio_62.mp3,c:\Users\victo\Desktop\Documents\Cuarto Año\TF...,You might find this lap that you meet a little...,5.088
4,driver_1,driver_1_belgium_radio_63.mp3,c:\Users\victo\Desktop\Documents\Cuarto Año\TF...,Just another two or three minutes to get throu...,5.712


## Filtering Out Post-Race Radio Messages

I need to eliminate the post-race radio messages that can bias the behaviour of our next models. 

### Identifying post-race messages

For identifying the post-race messages, these are some examples that can help detecting them:

* Congratulatory messages.
* Race result discussions.
* Thank you messages to the team.
* references to "the race" in past tense.
* Cool-down lap instructions.

--- 

### Code to filter messages

I will add a simple code function to try to detect this messages automatically. Then, after this test, I can proceed to delete the radio messages manually.


In [3]:
POST_RACE_KEYWORDS = [
    'race is over', 'good job', 'great race', 'next race',
    'cool down', 'cool-down', 'congratulations', 'result', 'finished'
]

def is_post_race(text):
    text_lower = text.lower()
    return any(kw in text_lower for kw in POST_RACE_KEYWORDS)

def flag_post_race_messages(df):
    df['is_post_race'] = df['text'].apply(is_post_race)
    return df


df = flag_post_race_messages(df)

In [4]:
def print_post_race_stats(df):
    count = df['is_post_race'].sum()
    print(f"Identified {count} potential post-race messages out of {len(df)} total")


print_post_race_stats(df)

Identified 34 potential post-race messages out of 659 total


In [5]:
def preview_post_race_messages(df, n=50):
    print("\nSample post-race messages:")
    for _, row in df[df['is_post_race']].head(n).iterrows():
        print(f"Driver {row['driver']}: {row['text']}")
        print("-" * 50)


preview_post_race_messages(df)


Sample post-race messages:
Driver driver_1: Yeah, I gave it all. I was a bit unlucky. Still I think some okay points I guess after difficult weekend. Yeah, well done Max. That was a really strong drive today. As GP said, we got had over by the safety car, but your comeback after the stop was very, very strong. So without the safety car, I think it could have been a better race for us today, but good job. Yeah, it was quite good. Quite cool. It's been a hell of a run and yeah, it was always going to come to a close at one point, but brush yourself down and go again next weekend. Yeah, it's okay. They can take one. We'll go again next week.
--------------------------------------------------
Driver driver_10: Yeah, I had this dust on my eye, like I've been crying the last 20 laps driving one, with one eye. Not the easiest. Not bad for one eye though. Someone will give you some drops. If you see me crying, it's not because I'm emotional for P7. Alright, alright, you won't tell anyone. Goo

In [6]:
def print_review_note():
    print("NOTE: This automatic detection is just a first pass.")
    print("Proceed to manual review before filtering.")


print_review_note()

NOTE: This automatic detection is just a first pass.
Proceed to manual review before filtering.


### Manual Review and Filtering

After manual review, 49 radios are going to be deleted. Therefore, I think it is a good idea to **add more radios, at least 3 races more, to have a good amount of radios**. After that, I will proceed to make again a manual analysis and eliminate the radios.

### More steps: eliminating columns that are irrelevant for data labeling.

For sentiment analysis, columns like **file_path, duration, filename and post_race** are not relevant for sentiment analysis. Therefore, I will eliminate them from the dataframe and generate a csv with this filtered radios.

Moreover, I will change the name of the column from "text" to "radio_message".

In [7]:
def prepare_for_labeling(df, proc_dir):
    columns_to_drop = ["is_post_race", "file_path", "duration", "filename"]
    df_out = df.drop(columns=columns_to_drop)
    df_out = df_out.rename(columns={'text': 'radio_message'})
    df_out.to_csv(proc_dir / "radios_2_columns.csv", index=False)
    return df_out


df_filtered = prepare_for_labeling(df, PROC_DIR)

--- 

## Sentiment Labeling

#### Creating the sentiment column (empty)

In [8]:
def init_sentiment_column(df):
    df = df.copy()
    df["sentiment"] = ""
    return df


df_filtered = init_sentiment_column(df_filtered)
df_filtered.head()

,driver,radio_message,sentiment
0,driver_1,"So don't forget Max, use your head please. Are...",
1,driver_1,"Okay Max, we're expecting rain in about 9 or 1...",
2,driver_1,$%&&S& $%&!,
3,driver_1,You might find this lap that you meet a little...,
4,driver_1,Just another two or three minutes to get throu...,


### Labeling the data manually


In [9]:
# ─── INTERACTIVE LABELING WIDGET ────────────────────────────────────────────
# The manual sentiment labeling interface (ipywidgets) has already been run.
# Labeled outputs exist in PROC_DIR:
#   - radio_labeled_data.csv  (all messages with sentiment labels)
#   - radio_filtered.csv      (post-race messages removed)
#
# Only re-run the widget if relabeling from scratch (requires running N18 first
# to produce radios_raw.csv). The full widget code is preserved in:
#   legacy/notebooks/NLP_radio_processing/N00_data_labeling.ipynb
# ─────────────────────────────────────────────────────────────────────────────

In [10]:
def clean_labeled_data(proc_dir):
    df = pd.read_csv(proc_dir / "radio_labeled_data.csv")
    original_length = len(df)
    print(f"Original dataset size: {original_length} rows")

    null_count = df['sentiment'].isna().sum()
    print(f"Empty values in the 'sentiment' column: {null_count}")

    df_clean = df.dropna(subset=['sentiment'])
    removed_count = original_length - len(df_clean)
    print(f"Rows removed: {removed_count}")
    print(f"New dataset size: {len(df_clean)} rows")

    df_clean.to_csv(proc_dir / "radio_labeled_data.csv", index=False)
    print(f"\nFile successfully saved at: {proc_dir / 'radio_labeled_data.csv'}")
    return df_clean


df_radio_labeled = clean_labeled_data(PROC_DIR)

Original dataset size: 530 rows
Empty values in the 'sentiment' column: 0
Rows removed: 0
New dataset size: 530 rows

File successfully saved at: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\radio_nlp\radio_labeled_data.csv


## Conclusions

This notebook produces:
- `radio_labeled_data.csv` — full dataset with sentiment labels and post-race messages removed.
- `radio_filtered.csv` — same dataset without the post-race messages (no sentiment column).

Next steps:
1. **N19** (`N19_sentiment_vader.ipynb`) — run VADER on the cleaned data as a baseline.
2. **N20** (`N20_bert_sentiment.ipynb`) — fine-tune RoBERTa for sentiment classification.